# Week 11, ZoroEval: the reusable eval harness

# Requirements: pip install openai numpy pandas

# ⚠️ REQUIRES: API key for judge cells (set OPENAI_API_KEY)

This notebook builds **ZoroEval**, the reusable harness that gates every ZoroLogistics AI
artifact. It has three golden sets (BoL extraction, RAG groundedness, ticket triage),
code metrics that need no key, and an **LLM-as-judge** with an anchored 1 to 5 rubric. It
then measures **judge agreement** on a doubled sample and **calibrates** the rubric
against human labels. The code-metric cells run without a key; the judge cells skip
gracefully when `OPENAI_API_KEY` is unset.


## 0. Setup: repo root on the path + seeded RNG


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root

import os, json, random, statistics
import numpy as np
import pandas as pd

from zoro import data

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    from openai import OpenAI
    _OPENAI_OK = True
except Exception as _e:  # pragma: no cover
    OpenAI = None
    _OPENAI_OK = False
    print("openai SDK not installed:", _e)

print("imports ok, openai SDK:", _OPENAI_OK)


## 1. Judge credentials (graceful skip)

The judge is a *different, usually stronger* model that grades open-ended answers.
Keys come from the environment, never hard-coded.


In [ ]:
API_KEY = os.environ.get("OPENAI_API_KEY")
JUDGE_MODEL = os.environ.get("ZOROEVAL_JUDGE_MODEL", "gpt-4o-mini")

judge_client = None
if API_KEY and _OPENAI_OK:
    judge_client = OpenAI(api_key=API_KEY)
    print("judge client ready (model:", JUDGE_MODEL, ")")
else:
    print("No OPENAI_API_KEY set, judge cells will skip; code-metric cells still run.")


## 2. Golden set (a): BoL field extraction

Ground truth comes from `data.bol_samples()`' `fields` dict. Extraction is graded by
**field-level exact match**: a correct date but wrong weight is still a failure, so an
"overall accuracy" would hide it.


In [ ]:
extraction_gold = data.bol_samples(n=20, seed=5)
print("extraction golden set:", len(extraction_gold))
print("fields to check:", sorted(extraction_gold[0]["fields"].keys()))


## 3. Golden set (b): RAG groundedness (10 Q/A pairs)

Groundedness is judged against the retrieved passage, so each pair carries the source
`doc_id`(s) that justify the answer. Ten pairs over `data.policy_docs()`, the same
policy corpus Week 7 used.


In [ ]:
docs = data.policy_docs()
by_id = {d["doc_id"]: d for d in docs}

rag_gold = [
    {"q": "How late does a shipment have to be to get a 10% freight refund?",
     "answer": "More than 48 hours late.", "sources": ["POL-002"]},
    {"q": "What does an address change cost after pickup?",
     "answer": "$85.", "sources": ["POL-001"]},
    {"q": "How long do I have to file a damage claim?",
     "answer": "Within 7 days of delivery.", "sources": ["POL-002"]},
    {"q": "What is the maximum refund for an approved damage claim?",
     "answer": "Up to $5,000.", "sources": ["POL-002"]},
    {"q": "Which items cannot be shipped at all?",
     "answer": "Lithium batteries over 100 Wh, explosives, and unapproved chemicals.", "sources": ["POL-003"]},
    {"q": "What documents does a dangerous-goods shipment need?",
     "answer": "A signed shipper's declaration and a UN number on the bill of lading.", "sources": ["POL-003"]},
    {"q": "Who pays duties and taxes on a cross-border shipment?",
     "answer": "The consignee, unless prepaid at booking.", "sources": ["POL-004"]},
    {"q": "What happens when customs holds a shipment beyond 5 days?",
     "answer": "A $40/day storage fee applies.", "sources": ["POL-004"]},
    {"q": "How much is the refund when a shipment is more than 7 days late?",
     "answer": "50% of the freight.", "sources": ["POL-002"]},
    {"q": "Are delivery SLAs extended during severe weather?",
     "answer": "Yes, by 48 hours without penalty.", "sources": ["POL-001"]},
]
print("RAG groundedness golden set:", len(rag_gold), "Q/A pairs")


## 4. Golden set (c): ticket triage

Ground truth is the `category` column. Graded by plain **accuracy** on the category.


In [ ]:
triage_gold = data.support_tickets(n=30, seed=99)
CATEGORIES = ["tracking", "damage", "refund", "documents", "customs", "billing"]
print("triage golden set:", len(triage_gold))
print(triage_gold["category"].value_counts().to_string())


## 5. The rubric (anchored, one dimension)

"Rate 1 to 5" produces noise; this rubric scores **groundedness only** and names the
concrete failure each level catches, a hallucinated rate, an unsupported date. That is
what makes the judge's output defensible in a release review.


In [ ]:
GROUNDEDNESS_RUBRIC = """Score the ANSWER for GROUNDEDNESS only, not helpfulness, not tone.

5, Every factual claim in the ANSWER is directly supported by the RETRIEVED PASSAGE,
    and the ANSWER cites the passage that supports it.
3, The core answer is supported, but it includes at least one claim (a number, date,
    or policy detail) not present in the passage.
1, The ANSWER states a fact that contradicts the passage, or invents a rate, date, or
    contract term the passage does not contain.
"""

def make_reference(pair, docs_by_id):
    return "\n\n".join(f"[{sid}] {docs_by_id[sid]['text']}" for sid in pair["sources"])


## 6. The ZoroEval class

One small class, three responsibilities: (1) code metrics that run fast and free,
(2) an LLM-as-judge method with the rubric, (3) a run log so every number is traceable.
This is the artifact later weeks import and extend.


In [ ]:
class ZoroEval:
    """Minimal reusable eval harness: golden sets + code metrics + LLM-as-judge."""

    def __init__(self, judge_client=None, judge_model="gpt-4o-mini"):
        self.judge_client = judge_client
        self.judge_model = judge_model
        self.runs = []  # (metric_name, value) log

    # ---- code metrics (no judge, no key) ----
    def extraction_field_accuracy(self, predictions, gold):
        fields = ["shipper", "port_of_loading", "port_of_discharge", "commodity",
                  "gross_weight_kg", "freight_terms"]
        total, correct = 0, 0
        for pred, ref in zip(predictions, gold):
            for k in fields:
                total += 1
                correct += (str(pred.get(k, "")).strip().lower()
                            == str(ref["fields"][k]).strip().lower())
        acc = correct / max(total, 1)
        self.runs.append(("extraction_field_accuracy", acc))
        return acc

    def triage_accuracy(self, predicted_labels, gold_labels):
        correct = sum(1 for p, g in zip(predicted_labels, gold_labels) if p == g)
        acc = correct / max(len(gold_labels), 1)
        self.runs.append(("triage_accuracy", acc))
        return acc

    # ---- LLM-as-judge (needs a client) ----
    def judge_groundedness(self, question, answer, reference):
        if self.judge_client is None:
            return None
        prompt = (
            f"{GROUNDEDNESS_RUBRIC}\n\n"
            f"QUESTION: {question}\n"
            f"ANSWER: {answer}\n"
            f"RETRIEVED PASSAGE:\n{reference}\n\n"
            f"Return a single integer score (1, 3, or 5)."
        )
        resp = self.judge_client.chat.completions.create(
            model=self.judge_model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
        )
        raw = resp.choices[0].message.content.strip()
        for tok in raw.split():
            if tok.isdigit():
                return int(tok)
        return None


## 7. Demonstrate the code metrics (no key needed)

A metric is only real if it can tell good from bad. We grade the ground-truth labels
("perfect") against a seeded-noise copy, and watch the number drop.


In [ ]:
ze = ZoroEval(judge_client=judge_client, judge_model=JUDGE_MODEL)

# triage: ground truth vs seeded 20% label noise
gold_labels = triage_gold["category"].tolist()
rng = np.random.default_rng(7)
noise_labels = [g if rng.random() > 0.2 else rng.choice(CATEGORIES) for g in gold_labels]

perfect_acc = ze.triage_accuracy(gold_labels, gold_labels)
noisy_acc = ze.triage_accuracy(noise_labels, gold_labels)
print(f"triage accuracy: perfect={perfect_acc:.3f}  noisy={noisy_acc:.3f}")

# extraction: ground-truth fields vs a copy with every weight corrupted by +1000
perfect_preds = [dict(b["fields"]) for b in extraction_gold]
corrupt_preds = []
for b in extraction_gold:
    p = dict(b["fields"])
    p["gross_weight_kg"] = int(p["gross_weight_kg"]) + 1000
    corrupt_preds.append(p)

print(f"extraction field accuracy: perfect={ze.extraction_field_accuracy(perfect_preds, extraction_gold):.3f}  "
      f"corrupted={ze.extraction_field_accuracy(corrupt_preds, extraction_gold):.3f}")


## 8. Judge agreement on a doubled sample

A judge is itself a model, so it has noise. We grade the *same* content twice on the
doubled sample and measure how often the judge agrees with itself. Low self-agreement
means the rubric is too vague to trust.


In [ ]:
def judge_agreement(zoroeval, rag_gold, docs_by_id, n_repeats=2):
    if zoroeval.judge_client is None:
        return None, []
    scores = []
    for pair in rag_gold:
        ref = make_reference(pair, docs_by_id)
        s1 = zoroeval.judge_groundedness(pair["q"], pair["answer"], ref)
        s2 = zoroeval.judge_groundedness(pair["q"], pair["answer"], ref)
        if s1 is not None and s2 is not None:
            scores.append((s1, s2))
    agree = sum(1 for a, b in scores if a == b) / max(len(scores), 1)
    return agree, scores


In [ ]:
agree, agree_scores = judge_agreement(ze, rag_gold, by_id)
if agree is None:
    print("judge agreement: skipped (no API key)")
else:
    print(f"judge agreement on {len(agree_scores)} doubled samples: {agree:.3f}")
    print("sample (run1, run2) score pairs:", agree_scores[:5])


## 9. Calibrate the rubric

Self-agreement is necessary but not sufficient, the judge could be consistently *wrong*.
Calibration compares the judge to **your** labels. Trust the judge only on dimensions
where its agreement with you is high; below that, fall back to human review or a code
metric. (`HUMAN_LABELS` are your grades for the 10 RAG answers, in order, change them to
*your* judgment and re-run.)


In [ ]:
HUMAN_LABELS = [5, 5, 5, 3, 5, 3, 5, 3, 3, 5]  # your labels for rag_gold, in order

def calibrate(zoroeval, rag_gold, docs_by_id, human_labels):
    if zoroeval.judge_client is None:
        return None, []
    judge_scores = []
    for pair in rag_gold:
        ref = make_reference(pair, docs_by_id)
        judge_scores.append(zoroeval.judge_groundedness(pair["q"], pair["answer"], ref))
    pairs = [(j, h) for j, h in zip(judge_scores, human_labels) if j is not None]
    agree = sum(1 for j, h in pairs if abs(j - h) <= 1) / max(len(pairs), 1)
    return agree, pairs


In [ ]:
cal_agree, cal_pairs = calibrate(ze, rag_gold, by_id, HUMAN_LABELS)
if cal_agree is None:
    print("calibration: skipped (no API key)")
else:
    print(f"calibration agreement (judge vs human, within +/-1): {cal_agree:.3f}")
    print("(judge, human) pairs:", cal_pairs)


## 10. Full judge pass over the RAG set

One groundedness score per Q/A pair. This is the per-example output that feeds the
error-analysis workshop (notebook 02): which pairs score 3 or 1 tell you *where* the
retrieval/generation is breaking.


In [ ]:
full_scores = []
if ze.judge_client is not None:
    for pair in rag_gold:
        ref = make_reference(pair, by_id)
        full_scores.append(ze.judge_groundedness(pair["q"], pair["answer"], ref))
    print("groundedness scores (1-5) over 10 Q/A pairs:", full_scores)
    valid = [s for s in full_scores if s is not None]
    print("mean groundedness:", round(statistics.mean(valid), 2))
else:
    print("full judge pass skipped (no API key).")


## 11. Takeaway

The harness is the point, not any single run. Code metrics cover what is deterministic
(extraction fields, triage labels); the judge covers the open-ended dimension
(groundedness), and only *after* agreement + calibration say it is trustworthy. A judge
you have not calibrated is a second unverified model.


In [ ]:
# FINAL number: judge self-agreement on the doubled sample (0.0 = judge did not run).
final_agree = agree if agree is not None else 0.0
print(f"JUDGE_AGREEMENT={final_agree:.3f}")
print(f"CALIBRATION_AGREEMENT={cal_agree if cal_agree is not None else 'skipped'}")
print(f"TRIAGE_ACC_NOISY={noisy_acc:.3f}   (code metric that always runs)")
